# Protecting Shared State

You will explain competing updates and protect shared Java state with a consistent synchronization rule.

You will trace how two workers can reach the same changing record, protect each complete update, and decide when its total is ready to report. A controlled sequential model explains a lost update without requiring a real race to produce a particular result.

## Learning Goals

- Explain a lost update using a specific read-compute-write interleaving.
- Protect a shared counter and a check-then-reserve operation with synchronized instance methods.

## Why This Matters

Two agents may refer to the same reservation record. Hiding its field inside a class does not stop their method calls from overlapping. Protecting the complete check and change preserves the record’s rule, such as never reserving the same last seat twice. Correctness depends on the shared operation and common lock, not on which caller usually finishes first.

Applications often maintain a shared count, inventory, or reservation record while handling independent requests. A reliable update rule lets those requests preserve the record’s meaning. This builds on starting and joining workers: waiting establishes completion, while synchronization protects the operations that occur before completion.

## Check Your Starting Point

Recall why a normally returning join permits the caller to read a completed worker’s result. Explain the difference between two objects and two references to one object.

In [ ]:
Your response:


<details>
<summary>Show answer</summary>

<a id="starting-point-interpretation"></a>

Each started worker’s normal untimed join establishes its completion and visibility to the joining caller. Shared references identify one object; constructing another object creates separate state.

</details>

## Video Demonstration

Watch how two jobs use one protected counter and wait for their completed shared total.

<video controls preload="metadata" width="960" aria-label="Protecting shared Java state demonstration">
<source src="media/02_protecting_shared_state/demo.mp4" type="video/mp4">
<track kind="captions" src="media/02_protecting_shared_state/captions.vtt" srclang="en" label="English">
</video>

[Read the protecting shared state video transcript.](media/02_protecting_shared_state/transcript.md)

## Concept

### Give two workers one shared record

A campus service desk wants to count requests handled by two workers. In our model, the first worker records 2,000 requests and the second records 3,000. Each request contributes one to a single total. The loops model those updates; they do not contact real customers. Success means reporting 5,000 after both workers finish, regardless of which worker makes progress first.

The previous lesson gave each worker its own changing count. This lesson deliberately changes that arrangement. **Shared mutable state** is data that more than one thread can access and that can change. Both workers will update one `SharedCounter` object.

```java
SharedCounter counter = new SharedCounter();
Thread first = new Thread(new IncrementJob(counter, 2000));
Thread second = new Thread(new IncrementJob(counter, 3000));
```

This fragment assumes the class declarations explained below. The first line creates exactly one counter. Both `IncrementJob` constructions receive the reference named `counter`, so their separate job objects lead back to that same counter. The numbers tell each job how many increments to request. Creating a new counter for each worker would create two separate records instead of the combined record we need.

Making a field `private` controls access from source code. It does not prevent two workers from entering public methods on the same object. We need an additional rule for their overlapping updates.

### Recognize an unordered conflict

A **data race** occurs when separate threads make conflicting accesses to the same variable, at least one access writes it, and the accesses lack the required ordering between threads. Two workers changing one unprotected count create this concern. Reading that changing count without a suitable ordering rule can also conflict with a write.

A familiar-looking assignment can hide several actions. To evaluate `total = total + 1`, a worker reads the old value, calculates a new value, and stores that new value. Another worker may access the same field between those actions. One source line is not automatically one indivisible operation across threads.

Starting both threads and joining them at the end establishes when their work is complete for the caller. It does not prevent their updates from interfering while they run. A correct final total from one unprotected run would not prove the update rule safe. We will establish a rule that protects every increment instead of relying on a favorable schedule.

### Trace how one update can disappear

A **lost update** happens when one update overwrites the effect of another. Imagine the counter currently holds seven. Worker A reads seven and prepares eight. Before A stores its result, worker B also reads seven and prepares eight. A stores eight; B then stores eight as well.

Two requests should have taken the count from seven to nine. Instead, the stored value is eight. Each addition was correct for the old value that worker read, but the combined operation lost one request. The problem is the overlap between reading and storing, not the arithmetic itself.

This is an illustrative ordering, not a recording of two actual threads. Other schedules can produce different observations. A later support program models the separate steps in an ordinary sequential program so you can trace their values directly. We use that controlled model to explain the failure mechanism without expecting a real race to produce a particular output on demand.

### Protect the whole state change

An **atomic state transition** is a change that other cooperating operations cannot observe or interrupt partway through. For this counter, the protected transition includes the read, addition, and write. Protecting only the final store would leave the earlier read exposed to another update.

The same reasoning applies to reservations, which you will practice later. If a ticket bank has one ticket left, checking that a ticket remains and subtracting the ticket must stay together. Otherwise, two callers could each pass the check before either changes the count. The rule belongs around the complete decision and change.

Atomicity here describes the operation relative to other callers following the same protection rule. It does not mean that a processor performs all of its instructions at once. Java provides synchronization so we can keep the operation together even though its work takes several steps.

### Use the counter object's monitor

Every Java object has an **intrinsic monitor**, a lock that synchronized code can use to coordinate access. A synchronized instance method uses the monitor belonging to the object on which that method was called. Here is the complete counter class from our example:

```java
class SharedCounter {
    private int total;
    public SharedCounter() { total = 0; }
    public synchronized void increment() { total = total + 1; }
    public synchronized int getTotal() { return total; }
}
```

The constructor begins the count at zero. The keyword **`synchronized`** on `increment` requires the caller to acquire this counter's monitor before entering the method. The method then performs the complete increment while holding that monitor. Java releases the monitor when the synchronized method exits, including when it exits because of an exception.

The getter uses the same rule when it reads the field. Both methods belong to the same receiver object, so both use the same monitor. We do not need separate instructions to acquire and release it in these method bodies.

The shared object is essential. Two different `SharedCounter` objects have different fields and different monitors. Synchronizing a method on one object does not protect unrelated accesses to another object, and an unprotected method would not automatically wait just because another method is synchronized.

### Let one protected operation finish before another enters

**Mutual exclusion** means that only one thread at a time can hold a given monitor. While one worker executes this counter's synchronized `increment`, another worker trying to enter a synchronized method on that same counter must wait. After the first releases the monitor, another caller can acquire it and perform its operation.

In the earlier seven-count illustration, the second protected increment cannot read the old seven while the first protected increment is still changing it. After the first increment completes, the next one can use eight and store nine. The complete read-compute-write sequence stays together relative to the other synchronized calls.

The `IncrementJob` loop requests one protected increment per iteration:

```java
for (int index = 0; index < count; index++) { counter.increment(); }
```

This fragment belongs inside the job's `run` method. `count` is that job's requested number of updates, and `counter` refers to the shared object. Each call acquires and releases the monitor for one increment. A worker does not hold the monitor for its entire loop, so calls from the two jobs can be interleaved between completed increments.

Mutual exclusion does not promise alternation, fairness, or a particular first worker. Our result depends on preserving every increment, not on choosing which worker goes next.

### Observe changes and then report completed work

Synchronization also provides **visibility**: releasing a monitor and later acquiring that same monitor establishes an ordering that lets the later caller observe the earlier protected writes. Mutual exclusion protects the operation from overlapping protected operations; this visibility rule lets the operations pass their results forward correctly.

All participating accesses must follow a consistent rule. In `SharedCounter`, both the update and getter use the same object's monitor. The final report also needs a completion rule, because a synchronized getter can read a valid intermediate count while workers still have increments left to perform.

```java
first.start();
second.start();
first.join();
second.join();
System.out.println("Total: " + counter.getTotal());
```

This fragment follows construction of both workers and the shared counter. Both starts come before either join, allowing both workers to make progress. Each untimed join targets a started worker. Normal return establishes that worker's termination and makes its earlier writes visible to the caller. Only after both joins does the caller request the final count.

The first job contributes 2,000 increments and the second contributes 3,000. The protected operation preserves all of them, so the completed program reports `Total: 5000`. Synchronization protects each update while the workers run; joining both workers establishes that no requested updates remain before this final report. These rules answer different questions, and this example needs both.

## Worked Example

A campus service desk combines request counts: one worker records 2,000 requests and another records 3,000 in one shared counter. TicketBank is a later transfer task.

```java
class SharedCounter {
    private int total;
    public SharedCounter() { total = 0; }
    public synchronized void increment() { total = total + 1; }
    public synchronized int getTotal() { return total; }
}
class IncrementJob implements Runnable {
    private SharedCounter counter;
    private int count;
    public IncrementJob(SharedCounter counter, int count) {
        this.counter = counter;
        this.count = count;
    }
    @Override
    public void run() {
        for (int index = 0; index < count; index++) { counter.increment(); }
    }
}
SharedCounter counter = new SharedCounter();
Thread first = new Thread(new IncrementJob(counter, 2000));
Thread second = new Thread(new IncrementJob(counter, 3000));
first.start();
second.start();
first.join();
second.join();
System.out.println("Total: " + counter.getTotal());
```

Expected output:

```text
Total: 5000
```

The complete program constructs one SharedCounter and passes that same reference to two IncrementJob objects. One job requests 2,000 increments and the other requests 3,000. Each loop iteration calls counter.increment, so the operation guarded by the counter's monitor is one increment rather than the entire job.

The caller starts both workers, joins both, and then calls the synchronized getter on the shared counter. The protected increments preserve every requested update, while the joins establish that both jobs have completed before the report. Together these rules account for Total: 5000. Neither one worker's first turn nor alternation between workers is part of that guarantee.

This service-desk model gives us a combined record to inspect. In guided practice, you will complete or change this counting program and diagnose references that lead to separate counters. Ticket reservations come later as a distinct application of protecting a complete decision and state change.

## Guided Practice

Use the explained program first to retrieve its reasoning, then apply the ideas to distinct completion, modification and debugging tasks.

Recall the already demonstrated result and explain how the program produces it. Identify the relevant owned or shared state and the rule that allows its final observation. Then run the complete example and preserve this response; this is retrieval, not an unseen prediction.

In [ ]:
Your response:


In [ ]:
class SharedCounter {
    private int total;
    public SharedCounter() { total = 0; }
    public synchronized void increment() { total = total + 1; }
    public synchronized int getTotal() { return total; }
}
class IncrementJob implements Runnable {
    private SharedCounter counter;
    private int count;
    public IncrementJob(SharedCounter counter, int count) {
        this.counter = counter;
        this.count = count;
    }
    @Override
    public void run() {
        for (int index = 0; index < count; index++) { counter.increment(); }
    }
}
SharedCounter counter = new SharedCounter();
Thread first = new Thread(new IncrementJob(counter, 2000));
Thread second = new Thread(new IncrementJob(counter, 3000));
first.start();
second.start();
first.join();
second.join();
System.out.println("Total: " + counter.getTotal());

Record the actual output from this run. Compare them with the already explained result. Identify any difference without changing your original retrieval response.

In [ ]:
Your response:


Trace one complete guarded increment, the common receiver and the caller’s joins. Explain why the final sum does not reveal the schedule.

In [ ]:
Your response:


<details>
<summary>Show answer</summary>

<a id="worked-interpretation"></a>

Both jobs update one receiver under its monitor. The two finite counts supply the expected sum; both joins finish before the getter reports it. A correct observed sum does not reveal a schedule or justify removing protection.

</details>

<details id="animation-shared_monitor_transition" class="animation-panel" open>
<summary>Protect each update; wait for the total — show or hide animation</summary>
<p><img src="media/02_protecting_shared_state/shared_monitor_transition.gif" alt="Both jobs refer to one counter. An illustrative protected call changes zero to one; a later protected call changes one to two. The remaining work is summarized. Both joins complete before the caller prints Total: 5000." width="960" style="max-width:100%;height:auto;"></p>
</details>

Each synchronized call uses the same counter monitor and protects the complete read, add, and write. The short sequence is an illustration, not an observed schedule. The tested final total combines 2,000 and 3,000 preserved increments after both joins. This silent loop lasts 22 seconds. Hide the animation to remove visible motion.

[View this final state as a still image](media/02_protecting_shared_state/shared_monitor_transition_still.png).


<a id="animation-shared_monitor_transition-still"></a>

[View the final state as a still image](media/02_protecting_shared_state/shared_monitor_transition_still.png).


### Model two overlapping increments

The next complete program is a sequential model: one ordinary execution stores two saved reads before performing either write. It makes the values visible without depending on an uncontrolled thread schedule. Read its code without running it, then write the prediction in the response area immediately below the prompt. The model starts at zero, while the earlier illustration started at seven; the same read-compute-write mechanism applies.

Trace the two saved reads and subsequent writes in this sequential lost-update model before running it. Predict its printed result and distinguish this model from an uncontrolled concurrent test.

In [ ]:
Your response:


In [ ]:
int shared = 0;
int firstRead = shared;
int secondRead = shared;
shared = firstRead + 1;
shared = secondRead + 1;
System.out.println("Both reads: " + firstRead + ", " + secondRead);
System.out.println("Modeled final value: " + shared);

Record the model’s actual result. Identify the old value used by each calculation and explain why the writes do not accumulate two independent increments.

In [ ]:
Your response:


<details>
<summary>Show answer</summary>

<a id="support-lost-update-answer-source-1"></a>

```java
int shared = 0;
int firstRead = shared;
int secondRead = shared;
shared = firstRead + 1;
shared = secondRead + 1;
System.out.println("Both reads: " + firstRead + ", " + secondRead);
System.out.println("Modeled final value: " + shared);
```

Expected output:

```text
Both reads: 0, 0
Modeled final value: 1
```

<a id="support-lost-update-interpretation"></a>

Both modeled reads use the same old value, so their separate writes overwrite rather than accumulate. This sequential model illustrates a possible lost update; it is not an observed uncontrolled race.

<details id="animation-modeled_lost_update" class="animation-panel" open>
<summary>Model how two updates can lose one change — show or hide animation</summary>
<p><img src="media/02_protecting_shared_state/modeled_lost_update.gif" alt="A sequential model stores zero in firstRead and secondRead. The first write stores one; the second write also stores one using its earlier snapshot. The program prints Both reads: 0, 0 and Modeled final value: 1." width="960" style="max-width:100%;height:auto;"></p>
</details>

The two snapshots remain zero even after the shared value changes. Each replacement is computed from a stored snapshot. These ordinary sequential statements explain how a change can be lost; they do not measure an actual unsafe worker schedule. This silent loop lasts 19 seconds. Hide the animation to remove visible motion.


<a id="animation-modeled_lost_update-still"></a>

[View the final state as a still image](media/02_protecting_shared_state/modeled_lost_update_still.png).


</details>

### Complete the monitor rule

Two campus event assistants record completed check-ins in one shared total. One job records 2 arrivals and the other 4.

Both IncrementJobs receive the same SharedCounter. Each complete increment and the getter follow its receiver monitor rule. Start and join both before the report.

Replace the two GUARD markers with the taught method modifier. Explain which receiver object supplies the monitor, which statements form one increment, and why both jobs must receive the same counter reference. Predict the final Total line. Before running your completed program, check that both marked methods use the modifier and both jobs still use counter.

Replace only the two GUARD markers with the taught method modifier that establishes the receiver monitor rule. Preserve the counter argument, input values, method bodies, starts, joins and print.

Read this supplied source before writing your response, then use the Java editing cell for your complete solution.

```java
class SharedCounter {
    private int total;
    public SharedCounter() { total = 0; }
    public GUARD void increment() { total = total + 1; }
    public GUARD int getTotal() { return total; }
}
class IncrementJob implements Runnable {
    private SharedCounter counter;
    private int count;
    public IncrementJob(SharedCounter counter, int count) {
        this.counter = counter;
        this.count = count;
    }
    @Override
    public void run() {
        for (int index = 0; index < count; index++) { counter.increment(); }
    }
}
SharedCounter counter = new SharedCounter();
Thread first = new Thread(new IncrementJob(counter, 2));
Thread second = new Thread(new IncrementJob(counter, 4));
first.start();
second.start();
first.join();
second.join();
System.out.println("Total: " + counter.getTotal());
```

In [ ]:
Your response:


Record the actual Total line and compare it with your prediction. Identify the shared receiver and explain how its monitor protects the whole read/add/write operation. Explain why the caller joins both jobs before reporting; do not infer a fair worker schedule.

In [ ]:
Your response:


<details>
<summary>Show answer</summary>

<a id="guided-completion-answer-source-1"></a>

```java
class SharedCounter {
    private int total;
    public SharedCounter() { total = 0; }
    public synchronized void increment() { total = total + 1; }
    public synchronized int getTotal() { return total; }
}
class IncrementJob implements Runnable {
    private SharedCounter counter;
    private int count;
    public IncrementJob(SharedCounter counter, int count) {
        this.counter = counter;
        this.count = count;
    }
    @Override
    public void run() {
        for (int index = 0; index < count; index++) { counter.increment(); }
    }
}
SharedCounter counter = new SharedCounter();
Thread first = new Thread(new IncrementJob(counter, 2));
Thread second = new Thread(new IncrementJob(counter, 4));
first.start();
second.start();
first.join();
second.join();
System.out.println("Total: " + counter.getTotal());
```

Expected output:

```text
Total: 6
```

<a id="guided-completion-interpretation"></a>

Both methods use synchronized. Because both jobs call increment on the same receiver, they acquire the same intrinsic monitor. One complete read/add/write operation finishes before a competing increment guarded by that monitor enters. The two finite jobs contribute 2 and 4 updates, and the caller joins both before reading 6. The result follows the code's protection and completion rules, not a promised worker schedule.

</details>

### Change the shared workload

One campus check-in desk receives no arrivals while another receives 5. Adapt the already explained shared total program to the new workloads.

A job may perform zero increments; both jobs still share one protected counter and both worker lifecycles finish before reporting.

The supplied 2000/3000 program is a fully explained baseline. Change only the two IncrementJob counts to 0 and 5. Predict the new Total line. Explain why the zero-work case does not authorize removing synchronized or a join from the general program. Run the edited complete program and record its actual result.

Change only first job count 2000 to 0 and second 3000 to 5. Preserve one shared counter, all synchronized methods, both starts and both joins.

Read this supplied source before writing your response, then use the Java editing cell for your complete solution.

```java
class SharedCounter {
    private int total;
    public SharedCounter() { total = 0; }
    public synchronized void increment() { total = total + 1; }
    public synchronized int getTotal() { return total; }
}
class IncrementJob implements Runnable {
    private SharedCounter counter;
    private int count;
    public IncrementJob(SharedCounter counter, int count) {
        this.counter = counter;
        this.count = count;
    }
    @Override
    public void run() {
        for (int index = 0; index < count; index++) { counter.increment(); }
    }
}
SharedCounter counter = new SharedCounter();
Thread first = new Thread(new IncrementJob(counter, 2000));
Thread second = new Thread(new IncrementJob(counter, 3000));
first.start();
second.start();
first.join();
second.join();
System.out.println("Total: " + counter.getTotal());
```

In [ ]:
Your response:


In [ ]:
class SharedCounter {
    private int total;
    public SharedCounter() { total = 0; }
    public synchronized void increment() { total = total + 1; }
    public synchronized int getTotal() { return total; }
}
class IncrementJob implements Runnable {
    private SharedCounter counter;
    private int count;
    public IncrementJob(SharedCounter counter, int count) {
        this.counter = counter;
        this.count = count;
    }
    @Override
    public void run() {
        for (int index = 0; index < count; index++) { counter.increment(); }
    }
}
SharedCounter counter = new SharedCounter();
Thread first = new Thread(new IncrementJob(counter, 2000));
Thread second = new Thread(new IncrementJob(counter, 3000));
first.start();
second.start();
first.join();
second.join();
System.out.println("Total: " + counter.getTotal());

Record the edited program’s Total line. Explain the zero-work job’s contribution, identify the counter shared by both jobs, and explain why the general program retains synchronized methods and both joins even for this input.

In [ ]:
Your response:


<details>
<summary>Show answer</summary>

<a id="guided-modification-answer-source-1"></a>

```java
class SharedCounter {
    private int total;
    public SharedCounter() { total = 0; }
    public synchronized void increment() { total = total + 1; }
    public synchronized int getTotal() { return total; }
}
class IncrementJob implements Runnable {
    private SharedCounter counter;
    private int count;
    public IncrementJob(SharedCounter counter, int count) {
        this.counter = counter;
        this.count = count;
    }
    @Override
    public void run() {
        for (int index = 0; index < count; index++) { counter.increment(); }
    }
}
SharedCounter counter = new SharedCounter();
Thread first = new Thread(new IncrementJob(counter, 0));
Thread second = new Thread(new IncrementJob(counter, 5));
first.start();
second.start();
first.join();
second.join();
System.out.println("Total: " + counter.getTotal());
```

Expected output:

```text
Total: 5
```

<a id="guided-modification-interpretation"></a>

The first loop has no iterations; the second contributes five protected increments. Both still receive the same counter, and both workers complete before its result is read. This special workload does not establish that synchronization is unnecessary for other inputs; the preserved program must still support two active workers. The fixed fixture has no scheduler-dependent expected output.

</details>

### Repair the shared references

Two event assistants are intended to contribute to one coordinator report. The assistants record 3 and 4 check-ins. A developer supplies a program whose report must be checked.

The intended report must read the counter that both jobs update. Every shown increment already uses a synchronized method; do not remove that protection.

This complete diagnostic is finite and safe to run. Before running, predict its Total line and trace which counter object each IncrementJob receives and which object the final getter reads. Explain whether synchronized by itself makes those object references refer to one counter. Run the diagnostic and record its actual line before making the allowed repair.

After recording the diagnostic, change only the first argument expression of each IncrementJob constructor. Preserve the classes, input counts, starts, joins, report and every synchronized modifier. State the replacement expressions in your repair plan before editing.

Read this supplied source before writing your response. Run the separate diagnostic cell first, then use the later editing cell for your complete repair.

```java
class SharedCounter {
    private int total;
    public SharedCounter() { total = 0; }
    public synchronized void increment() { total = total + 1; }
    public synchronized int getTotal() { return total; }
}
class IncrementJob implements Runnable {
    private SharedCounter counter;
    private int count;
    public IncrementJob(SharedCounter counter, int count) {
        this.counter = counter;
        this.count = count;
    }
    @Override
    public void run() {
        for (int index = 0; index < count; index++) { counter.increment(); }
    }
}
SharedCounter counter = new SharedCounter();
Thread first = new Thread(new IncrementJob(new SharedCounter(), 3));
Thread second = new Thread(new IncrementJob(new SharedCounter(), 4));
first.start();
second.start();
first.join();
second.join();
System.out.println("Total: " + counter.getTotal());
```

In [ ]:
Your response:


In [ ]:
class SharedCounter {
    private int total;
    public SharedCounter() { total = 0; }
    public synchronized void increment() { total = total + 1; }
    public synchronized int getTotal() { return total; }
}
class IncrementJob implements Runnable {
    private SharedCounter counter;
    private int count;
    public IncrementJob(SharedCounter counter, int count) {
        this.counter = counter;
        this.count = count;
    }
    @Override
    public void run() {
        for (int index = 0; index < count; index++) { counter.increment(); }
    }
}
SharedCounter counter = new SharedCounter();
Thread first = new Thread(new IncrementJob(new SharedCounter(), 3));
Thread second = new Thread(new IncrementJob(new SharedCounter(), 4));
first.start();
second.start();
first.join();
second.join();
System.out.println("Total: " + counter.getTotal());


Record the actual diagnostic Total line. Compare it with your prediction and identify the counter object read by the final getter. Preserve this result when you repair the program.

In [ ]:
Your response:


State the two permitted argument replacements and predict the repaired Total line. Explain why the monitor and report must concern the same counter object. Then copy and repair the complete diagnostic in the separate work cell.

In [ ]:
Your response:


Record the repaired Total line alongside the preserved diagnostic result. Trace the objects updated and the object read before and after your repair. Explain why synchronization alone cannot combine separate counter objects, and why both jobs now contribute to the reported object.

In [ ]:
Your response:


<details>
<summary>Show answer</summary>

<a id="guided-debug-answer-source-1"></a>

```java
class SharedCounter {
    private int total;
    public SharedCounter() { total = 0; }
    public synchronized void increment() { total = total + 1; }
    public synchronized int getTotal() { return total; }
}
class IncrementJob implements Runnable {
    private SharedCounter counter;
    private int count;
    public IncrementJob(SharedCounter counter, int count) {
        this.counter = counter;
        this.count = count;
    }
    @Override
    public void run() {
        for (int index = 0; index < count; index++) { counter.increment(); }
    }
}
SharedCounter counter = new SharedCounter();
Thread first = new Thread(new IncrementJob(counter, 3));
Thread second = new Thread(new IncrementJob(counter, 4));
first.start();
second.start();
first.join();
second.join();
System.out.println("Total: " + counter.getTotal());
```

Expected output:

```text
Total: 7
```

<a id="guided-debug-interpretation"></a>

The diagnostic constructs three counters: the coordinator's counter and one private counter for each job. Each private counter is updated correctly under its own monitor, but neither job changes the object read by the report, so that report remains 0. The repair passes counter to both IncrementJobs. Their increments then target the same state and monitor; 3 plus 4 completed updates produce 7 after the joins. Synchronization protects access to an object; it does not combine separate objects or redirect references.

</details>

## Independent Practice

Build a complete program that transfers the taught mechanism to the following task. Keep the explained answer closed while planning, implementing and testing.

Two campus booking jobs must share a TicketBank initially holding five tickets. Write a synchronized reserve method that returns false at zero; otherwise decrease remaining once and return true. A synchronized getter reports remaining. Each TicketBuyer Runnable tries three reservations and keeps its own successes. Start and join both workers before printing their combined Reserved count and the bank’s Remaining count. Plan the shared monitor and predict the combined reports without promising an individual buyer split.

In [ ]:
Your response:


Record every actual baseline report. Compare it with your plan, explain the mechanism, and retain any discrepancy as evidence for a repair.

In [ ]:
Your response:


Before running any boundary variant, predict combined Reserved and Remaining for shared supplies 0 and 2 with three attempts per buyer. Then keep supply 5 and change only attempt counts from 3/3 to 1/6. Predict combined reports and explain why each buyer’s individual success count is not fixed. Preserve classes, shared receiver, starts, joins and reports.

In [ ]:
Your response:


Record the actual result for each named variant separately, preserving the predictions. Explain what boundary each checks and what it cannot establish about scheduling.

In [ ]:
Your response:


<details>
<summary>Show answer</summary>

<a id="independent-answer-source-1"></a>

### Baseline: five tickets and three attempts per buyer

The bank has five tickets and the buyers make six attempts altogether. The complete reservation rule permits five successes and leaves zero tickets. Each buyer can succeed at most three times; the combined report is fixed even though their split can differ.

```java
class TicketBank {
    private int remaining;
    public TicketBank(int remaining) { this.remaining = remaining; }
    public synchronized boolean reserve() {
        if (remaining == 0) { return false; }
        remaining--;
        return true;
    }
    public synchronized int getRemaining() { return remaining; }
}
class TicketBuyer implements Runnable {
    private TicketBank bank;
    private int attempts;
    private int successes;
    public TicketBuyer(TicketBank bank, int attempts) {
        this.bank = bank;
        this.attempts = attempts;
        this.successes = 0;
    }
    @Override
    public void run() {
        for (int index = 0; index < attempts; index++) {
            if (bank.reserve()) { successes++; }
        }
    }
    public int getSuccesses() { return successes; }
}
TicketBank bank = new TicketBank(5);
TicketBuyer firstBuyer = new TicketBuyer(bank, 3);
TicketBuyer secondBuyer = new TicketBuyer(bank, 3);
Thread first = new Thread(firstBuyer);
Thread second = new Thread(secondBuyer);
first.start();
second.start();
first.join();
second.join();
System.out.println("Reserved: " + (firstBuyer.getSuccesses() + secondBuyer.getSuccesses()));
System.out.println("Remaining: " + bank.getRemaining());
```

Expected output:

```text
Reserved: 5
Remaining: 0
```

<a id="independent-answer-source-2"></a>

### Empty supply: zero tickets

Every reservation finds zero remaining and returns false without decreasing it. The two buyers together reserve zero tickets, and the bank still contains zero.

```java
class TicketBank {
    private int remaining;
    public TicketBank(int remaining) { this.remaining = remaining; }
    public synchronized boolean reserve() {
        if (remaining == 0) { return false; }
        remaining--;
        return true;
    }
    public synchronized int getRemaining() { return remaining; }
}
class TicketBuyer implements Runnable {
    private TicketBank bank;
    private int attempts;
    private int successes;
    public TicketBuyer(TicketBank bank, int attempts) {
        this.bank = bank;
        this.attempts = attempts;
        this.successes = 0;
    }
    @Override
    public void run() {
        for (int index = 0; index < attempts; index++) {
            if (bank.reserve()) { successes++; }
        }
    }
    public int getSuccesses() { return successes; }
}
TicketBank bank = new TicketBank(0);
TicketBuyer firstBuyer = new TicketBuyer(bank, 3);
TicketBuyer secondBuyer = new TicketBuyer(bank, 3);
Thread first = new Thread(firstBuyer);
Thread second = new Thread(secondBuyer);
first.start();
second.start();
first.join();
second.join();
System.out.println("Reserved: " + (firstBuyer.getSuccesses() + secondBuyer.getSuccesses()));
System.out.println("Remaining: " + bank.getRemaining());
```

Expected output:

```text
Reserved: 0
Remaining: 0
```

<a id="independent-answer-source-3"></a>

### Limited supply: two tickets

Only two of the six attempts can succeed. Each successful reservation checks and decreases the same bank while holding its monitor, so the combined report is two reserved and zero remaining.

```java
class TicketBank {
    private int remaining;
    public TicketBank(int remaining) { this.remaining = remaining; }
    public synchronized boolean reserve() {
        if (remaining == 0) { return false; }
        remaining--;
        return true;
    }
    public synchronized int getRemaining() { return remaining; }
}
class TicketBuyer implements Runnable {
    private TicketBank bank;
    private int attempts;
    private int successes;
    public TicketBuyer(TicketBank bank, int attempts) {
        this.bank = bank;
        this.attempts = attempts;
        this.successes = 0;
    }
    @Override
    public void run() {
        for (int index = 0; index < attempts; index++) {
            if (bank.reserve()) { successes++; }
        }
    }
    public int getSuccesses() { return successes; }
}
TicketBank bank = new TicketBank(2);
TicketBuyer firstBuyer = new TicketBuyer(bank, 3);
TicketBuyer secondBuyer = new TicketBuyer(bank, 3);
Thread first = new Thread(firstBuyer);
Thread second = new Thread(secondBuyer);
first.start();
second.start();
first.join();
second.join();
System.out.println("Reserved: " + (firstBuyer.getSuccesses() + secondBuyer.getSuccesses()));
System.out.println("Remaining: " + bank.getRemaining());
```

Expected output:

```text
Reserved: 2
Remaining: 0
```

<a id="independent-answer-source-4"></a>

### Unequal attempts: one and six

Keep five tickets, but give the buyers one and six attempts. They can reserve all five tickets. The first buyer may succeed zero or one time, so the second succeeds five or four times; the combined report remains five and zero.

```java
class TicketBank {
    private int remaining;
    public TicketBank(int remaining) { this.remaining = remaining; }
    public synchronized boolean reserve() {
        if (remaining == 0) { return false; }
        remaining--;
        return true;
    }
    public synchronized int getRemaining() { return remaining; }
}
class TicketBuyer implements Runnable {
    private TicketBank bank;
    private int attempts;
    private int successes;
    public TicketBuyer(TicketBank bank, int attempts) {
        this.bank = bank;
        this.attempts = attempts;
        this.successes = 0;
    }
    @Override
    public void run() {
        for (int index = 0; index < attempts; index++) {
            if (bank.reserve()) { successes++; }
        }
    }
    public int getSuccesses() { return successes; }
}
TicketBank bank = new TicketBank(5);
TicketBuyer firstBuyer = new TicketBuyer(bank, 1);
TicketBuyer secondBuyer = new TicketBuyer(bank, 6);
Thread first = new Thread(firstBuyer);
Thread second = new Thread(secondBuyer);
first.start();
second.start();
first.join();
second.join();
System.out.println("Reserved: " + (firstBuyer.getSuccesses() + secondBuyer.getSuccesses()));
System.out.println("Remaining: " + bank.getRemaining());
```

Expected output:

```text
Reserved: 5
Remaining: 0
```

<a id="independent-interpretation"></a>

For the final unequal-attempts variant, there are seven attempts for five tickets. Each synchronized reserve call checks and changes the same bank while holding its monitor. Exactly five calls can succeed, leaving zero tickets. Both joins complete before the caller adds the job-owned success counts. The first buyer may reserve zero or one ticket; the second then reserves five or four. The combined result is fixed, but the split depends on which reservations occur first. For zero or two tickets, the complete check and decrement remain inside the same bank monitor; combined successes cannot exceed the available supply.

The animation returns to the original five-ticket case, with three attempts per buyer.

<details id="animation-reservation_boundary" class="animation-panel" open>
<summary>Keep the availability check and change together — show or hide animation</summary>
<p><img src="media/02_protecting_shared_state/reservation_boundary.gif" alt="Two buyers make three attempts each against one bank of five tickets. An illustrative protected call checks the last ticket, decreases remaining from one to zero, and returns true. A later call returns false without decreasing. Both joined buyers report Reserved: 5 and Remaining: 0." width="960" style="max-width:100%;height:auto;"></p>
</details>

Checking availability and decreasing remaining belong in the same protected transition. The empty-bank path skips the decrease and success return. The two buyers own separate success counts; after both joins, their combined result is five, while their individual split may vary. This silent loop lasts 22 seconds. Hide the animation to remove visible motion.


<a id="animation-reservation_boundary-still"></a>

[View the final state as a still image](media/02_protecting_shared_state/reservation_boundary_still.png).


</details>

## Summary

From memory, explain these lesson ideas in a connected account: Shared mutable state, Data race, Lost update, Atomic state transition, Intrinsic monitor, Mutual exclusion, Synchronization visibility. Include one limit of the example evidence.

In [ ]:
Your response:


<details>
<summary>Show answer</summary>

<a id="summary-retrieval-interpretation"></a>

Shared references lead to one changing record. Without an ordering rule, read/add/write actions can conflict and lose an update. Protect the complete transition with synchronized operations on the same receiver monitor. Mutual exclusion keeps cooperating operations apart; visibility passes their changes forward. Both joins also establish that the final report follows completed work.

</details>

## Reflection

Describe a campus or project task that could use this lesson’s mechanism. Identify the work, owned or shared state, completion rule and one limitation of the analogy. Explain what would fail if the rule were omitted.

In [ ]:
Your response:


The next lesson coordinates a handoff between a producer and a consumer. Protecting an update does not by itself arrange when a value becomes available.

## Supplemental Reading

- [Java language locks and synchronization](https://docs.oracle.com/javase/specs/jls/se21/html/jls-17.html#jls-17.1) explains monitors and synchronized methods.
- [Java language memory ordering](https://docs.oracle.com/javase/specs/jls/se21/html/jls-17.html#jls-17.4.5) defines ordering between monitor operations and after a completed join.
- [Java 21 Thread](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/lang/Thread.html) documents starts and the completion wait used by the caller.
- [Java 21 Runnable](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/lang/Runnable.html) defines the work method implemented by IncrementJob and TicketBuyer.
- [Synchronized methods](https://docs.oracle.com/javase/tutorial/essential/concurrency/syncmeth.html) is the assigned Oracle tutorial explanation. Its examples target an older Java release; the Java 21 language and API links above describe our runtime.